<a href="https://colab.research.google.com/github/NatalieAbel/Capstone-Project/blob/main/stable_diffusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import torch
from tqdm import tqdm
import diffusers
from diffusers import StableDiffusionXLPipeline
import os
import pandas as pd
import subprocess

from PIL import Image
from IPython.display import display
from transformers import AutoImageProcessor, AutoModel

import torchaudio
from transformers import AutoProcessor, HubertModel


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [3]:
#get sample of audiocaps dataset
df = pd.read_csv("audiocaps.csv")

#oversample since some YouTube clips fail
sample_df = df.sample(n=200, random_state=42).copy()
sample_df.to_csv("audiocaps_sample_200.csv", index=False)

In [4]:
pip install yt-dlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 79.6 MB/s eta 0:00:00


In [5]:
#generate audio of 200 sample dataset
sample_df = pd.read_csv("audiocaps_sample_200.csv")

audio_root = "audio"
temp_root = "temp_audio"
os.makedirs(audio_root, exist_ok=True)
os.makedirs(temp_root, exist_ok=True)

for _, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    audiocap_id = str(row["audiocap_id"])
    youtube_id = str(row["youtube_id"])
    start_time = float(row["start_time"])

    final_wav_path = os.path.join(audio_root, f"{audiocap_id}.wav")
    if os.path.exists(final_wav_path):
        continue

    youtube_url = f"https://www.youtube.com/watch?v={youtube_id}"
    temp_template = os.path.join(temp_root, f"{youtube_id}.%(ext)s")

    #download best audio only
    download_cmd = [
        "yt-dlp",
        "-f", "bestaudio",
        "-o", temp_template,
        youtube_url
    ]
    subprocess.run(download_cmd, capture_output=True, text=True)

    downloaded_path = None
    for ext in ["m4a", "webm", "mp3", "opus"]:
        sample = os.path.join(temp_root, f"{youtube_id}.{ext}")
        if os.path.exists(sample):
            downloaded_path = sample
            break

    if downloaded_path is None:
        continue

    #trim to the 10-second AudioCaps clip
    ffmpeg_cmd = [
        "ffmpeg",
        "-y",
        "-ss", str(start_time),
        "-i", downloaded_path,
        "-t", "10",
        "-ac", "1",
        "-ar", "16000",
        final_wav_path
    ]
    subprocess.run(ffmpeg_cmd, capture_output=True, text=True)

    #delete temp full audio file to save space
    if os.path.exists(downloaded_path):
        os.remove(downloaded_path)

100%|██████████| 200/200 [08:39<00:00,  2.60s/it]


In [6]:
#keep only rows whose audio exists and make 100-sample CSV
valid_rows = []

for _, row in sample_df.iterrows():
    audio_id = str(row["audiocap_id"])
    audio_path = os.path.join("audio", f"{audio_id}.wav")
    if os.path.exists(audio_path):
        valid_rows.append(row)

valid_df = pd.DataFrame(valid_rows)
valid_df.to_csv("audiocaps_audio_sample_200.csv", index=False)

print("Valid rows with audio:", len(valid_df))

final_df = valid_df.sample(n=100, random_state=42).copy()
final_df.to_csv("audiocaps_audio_sample_100.csv", index=False)

print("Final sample size:", len(final_df))

Valid rows with audio: 183
Final sample size: 100


In [ ]:
#generate large images for 100-sample subset

#load subset and stable diffusion pipeline
subset_df = pd.read_csv("audiocaps_sample_100_expanded.csv")

model_id = "stabilityai/stable-diffusion-xl-base-1.0"
device = "cuda" if torch.cuda.is_available() else "cpu"

#initialize the pipeline
pipe = StableDiffusionXLPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    use_safetensors=True
)
pipe = pipe.to(device)
pipe.enable_attention_slicing()

for _, row in tqdm(subset_df.iterrows(),
    total = len(subset_df)):
      caption = row["expanded_caption"]
      audio_id = str(row["audiocap_id"])
      output_dir = "images/stable_diffusion/"
      model_dir = os.path.join(output_dir, audio_id)
      if not os.path.isdir(model_dir):
        os.makedirs(model_dir, exist_ok=True)
      files = [f for f in os.listdir(model_dir)]
      prompt = 'Generate an image of: ' + caption
      if len(files) >= 1:
        continue
      else:
        print(prompt, len(files))
        with torch.no_grad():
            generator = torch.Generator(device=device).manual_seed(42)
            image = pipe(
                prompt=prompt + ", photorealistic, highly detailed, realistic photo, 8k, cinematic lighting, sharp focus",
                negative_prompt="drawing, sketch, painting, cartoon, abstract, illustration, low quality, blurry, deformed, artistic, stylized",
                generator=generator,
                num_inference_steps=40,
                guidance_scale=8.5
            ).images[0]
            image.save(os.path.join(model_dir, "0.png"))
print("Done generating images!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

...

Generate an image of: In a bustling street corner, a sleek, high-performance sports car lets out a series of sharp, high-pitched squeals as it speeds past, accompanied by the constant humming of its powerful engine. 0


  0%|          | 0/40 [00:00<?, ?it/s]

 29%|██▉       | 29/100 [24:40<1:00:50, 51.41s/it]The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['photorealistic, highly detailed, realistic photo, 8 k, cinematic lighting, sharp focus']
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['photorealistic, highly detailed, realistic photo, 8 k, cinematic lighting, sharp focus']


Generate an image of: In the scene, a rhythmic electronic dinging sound emanates from various electronic devices in the background, punctuated by a vehicle horn sounding two sharp honks, as a distinctive plastic click can be heard, possibly indicating a child's toy or a similar item being used nearby, and all the while a distant canine voice expresses itself through a series 0


  0%|          | 0/40 [00:00<?, ?it/s]

In [ ]:
#preview generated images
subset_df = pd.read_csv("audiocaps_sample_100.csv")

for _, row in subset_df.iterrows():
    audio_id = str(row["audiocap_id"])
    img_path = f"images/stable_diffusion_xl/{audio_id}/0.png"

    if os.path.exists(img_path):
        print("Audio ID:", audio_id)
        print("Caption:", row["caption"])
        img = Image.open(img_path)
        img = img.resize((150, 150))
        display(img)
        print("-" * 60)

In [ ]:
#extract DINOv2 (large) embeddings from images

device = "cuda" if torch.cuda.is_available() else "cpu"

#load images and DINOv2 model
subset_df = pd.read_csv("audiocaps_sample_100.csv")

image_root = "images/stable_diffusion_xl"
save_root = "images/image_embeddings/DINOV2_LARGE_SDXL"
os.makedirs(save_root, exist_ok=True)

model_name = "facebook/dinov2-large"

#load processor and model
processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

#extract embeddings
for _, row in tqdm(subset_df.iterrows(), total=len(subset_df)):
    audio_id = str(row["audiocap_id"])

    img_path = os.path.join(image_root, audio_id, "0.png")
    save_path = os.path.join(save_root, f"{audio_id}.npy")

    if not os.path.exists(img_path):
        print(f"Missing image for {audio_id}")
        continue

    if os.path.exists(save_path):
        continue

    image = Image.open(img_path).convert("RGB")

    inputs = processor(
        images=image,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

        cls_embedding = outputs.last_hidden_state[:, 0, :]
        embedding = cls_embedding.squeeze(0).cpu().numpy()

    np.save(save_path, embedding)

In [ ]:
#download image embeddings in zip file
!zip -r dinov2_large_embeddings_sdxl.zip images/image_embeddings/DINOV2_LARGE_SDXL
from google.colab import files
files.download("dinov2_large_embeddings_sdxl.zip")

In [ ]:
#extract DINOv2 (base) embeddings from images

device = "cuda" if torch.cuda.is_available() else "cpu"

#load images and DINOv2 model
subset_df = pd.read_csv("audiocaps_sample_100.csv")

image_root = "images/stable_diffusion_base"
save_root = "images/image_embeddings/DINOV2_BASE_SDXL"
os.makedirs(save_root, exist_ok=True)

model_name = "facebook/dinov2-base"

#load processor and model
processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

#extract embeddings
for _, row in tqdm(subset_df.iterrows(), total=len(subset_df)):
    audio_id = str(row["audiocap_id"])

    img_path = os.path.join(image_root, audio_id, "0.png")
    save_path = os.path.join(save_root, f"{audio_id}.npy")

    if not os.path.exists(img_path):
        print(f"Missing image for {audio_id}")
        continue

    if os.path.exists(save_path):
        continue

    image = Image.open(img_path).convert("RGB")

    inputs = processor(
        images=image,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

        cls_embedding = outputs.last_hidden_state[:, 0, :]
        embedding = cls_embedding.squeeze(0).cpu().numpy()

    np.save(save_path, embedding)

In [ ]:
#download image embeddings in zip file
!zip -r dinov2_large_embeddings_sdxl.zip images/image_embeddings/DINOV2_LARGE_SDXL
from google.colab import files
files.download("dinov2_base_embeddings_sdxl.zip")

In [ ]:
#extract HuBERT (large) embeddings from audio

device = "cuda" if torch.cuda.is_available() else "cpu"

#load audio and processor
subset_df = pd.read_csv("audiocaps_sample_100.csv")

audio_root = "audio"
save_root = "audio/audio_embeddings/HUBERT_LARGE"
os.makedirs(save_root, exist_ok=True)

model_name = "facebook/hubert-large-ls960-ft"

processor = AutoProcessor.from_pretrained(model_name)
model = HubertModel.from_pretrained(model_name).to(device)
model.eval()

#extract embeddings
for _, row in tqdm(subset_df.iterrows(), total=len(subset_df)):
    audio_id = str(row["audiocap_id"])

    audio_path = os.path.join(audio_root, f"{audio_id}.wav")
    save_path = os.path.join(save_root, f"{audio_id}.npy")

    if not os.path.exists(audio_path):
        print(f"Missing audio for {audio_id}")
        continue

    if os.path.exists(save_path):
        continue

    waveform, sr = torchaudio.load(audio_path)

    waveform = waveform.mean(dim=0)

    if sr != 16000:
        resampler = torchaudio.transforms.Resample(sr, 16000)
        waveform = resampler(waveform)

    inputs = processor(
        waveform,
        sampling_rate=16000,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

        hidden_states = outputs.last_hidden_state


        embedding = hidden_states.mean(dim=1).squeeze(0).cpu().numpy()

    np.save(save_path, embedding)

In [ ]:
#download audio embeddings in zip file
!zip -r hubert_large_audio_embeddings.zip audio/audio_embeddings/HUBERT_LARGE
from google.colab import files
files.download("hubert_large_audio_embeddings.zip")